In [19]:
import os
import cv2
import pandas as pd
from ultralytics import YOLO

VIDEO_PATH   = "data/video4.mp4"
OUTPUT_VIDEO = "outputs/hasil_tracking4.mp4"
OUTPUT_CSV   = "outputs/hasil_per_frame4.csv"
os.makedirs("outputs", exist_ok=True)

In [20]:
model = YOLO("yolov8s.pt")

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Resolusi {w}x{h}, {fps:.1f} fps, {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))} frame")
cap.release()

writer = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

Resolusi 320x240, 30.0 fps, 2031 frame


Deteksi, tracking, hitung objek, dan cari objek hilang

In [21]:
records, prev_ids, all_ids = [], set(), set()

results = model.track(
    source=VIDEO_PATH,
    tracker="bytetrack.yaml",
    conf=0.2,
    imgsz=640,
    persist=True,
    stream=True,
    verbose=False,
)

for frame_no, r in enumerate(results, start=1):
    curr_ids = set(r.boxes.id.int().tolist()) if r.boxes.id is not None else set()

    lost_ids = sorted(prev_ids - curr_ids)
    new_ids  = sorted(curr_ids - all_ids)
    all_ids |= curr_ids

    records.append({
        "frame": frame_no,
        "jumlah_objek": len(curr_ids),
        "id_objek": sorted(curr_ids),
        "id_hilang": lost_ids,
        "id_baru": new_ids,
    })

    writer.write(r.plot())
    prev_ids = curr_ids

writer.release()

In [22]:
df = pd.DataFrame(records)
df.to_csv(OUTPUT_CSV, index=False)

print("Total ID unik         :", len(all_ids))
print("Rata-rata objek/frame :", round(df["jumlah_objek"].mean(), 2))

df[["frame", "jumlah_objek", "id_objek", "id_hilang"]]

Total ID unik         : 129
Rata-rata objek/frame : 4.19


,frame,jumlah_objek,id_objek,id_hilang
0,1,2,"[1, 2]",[]
1,2,2,"[1, 2]",[]
2,3,2,"[1, 2]",[]
3,4,2,"[1, 2]",[]
4,5,4,"[1, 2, 3, 4]",[]
...,...,...,...,...
2026,2027,3,"[731, 753, 779]",[712]
2027,2028,3,"[731, 753, 779]",[]
2028,2029,3,"[731, 753, 779]",[]
2029,2030,3,"[731, 753, 779]",[]
